<a id="contents"></a>
# Computer Vision
## หลักการ ตัวอย่าง และแบบฝึกหัดด้วย AI

เอกสารอบรม Basic Image Processing และ Image Transformation อ้างอิงสไลด์ของ Asst. Prof. Dr. Anakkapon Saenthon, KMITL ใช้ภาพจากชุด dataset ประกอบการเรียน

อ่านเนื้อหาและภาพประกอบ ส่ง Prompt ใน “ตัวอย่างที่” ให้ AI แล้วนำโค้ดมารัน จากนั้นทำ “แบบฝึกหัดที่” ด้วยการตัดสินใจของตนเอง ผู้สอนตรวจความเข้าใจจากภาพและเหตุผลประกอบ

**การเขียน Prompt:** ใช้ภาษาไทยบอกเป้าหมาย ใช้ชื่อเทคนิคและไฟล์ตามจริง ระบุค่าตัวเลขพร้อมหน่วย เช่น “กว้าง 300 สูง 420 พิกเซล” ข้อความใน [วงเล็บเหลี่ยม] เป็นส่วนที่ต้องเติมก่อนส่งให้ AI ซึ่งถ้าต้องการกระซับสามารถใช้เป็นภาษาอังกฤษเพื่อใช้งานได้ประสิทธิ์ภาพที่สุด

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bluebox-dev/Prompt-Computer-Vision/blob/main/Computer_Vision_AI_Workshop.ipynb)

**ใช้บนเครื่อง:** เปิด Jupyter จากโฟลเดอร์ repository นี้แล้วรันเซลล์เตรียมข้อมูลเช่นเดียวกัน

**ภาพประกอบ:** โหลดจาก `assets/notebook/` บน GitHub ผ่านอินเทอร์เน็ต ผู้สอนต้องอัปโหลดทั้ง notebook, `dataset/` และ `assets/` ไปยัง repository สาธารณะตามลิงก์นี้ก่อน

**กิจกรรม:** โค้ดตัวอย่างทั้ง 8 บทพร้อมรัน สามารถลองส่ง Prompt ให้ AI แล้วเปรียบเทียบผลได้ ส่วนเซลล์แบบฝึกหัดเว้นไว้ให้ผู้เรียนทำเอง

**รูปแบบโค้ด:** ตัวอย่างกำหนด 25–35 บรรทัดในเซลล์เดียว ใช้คำสั่งพื้นฐาน หาก AI ตอบยาว ให้ส่ง “ย่อให้ตรงจำนวนบรรทัดเดิม หนึ่งคำสั่งต่อบรรทัด คงผลลัพธ์เดิม” หากรันผิดพลาด ให้แนบ Error แล้วขอแก้เฉพาะจุด

### สารบัญ
1. [ภาพดิจิทัลและระบบสี](#lesson-1)
2. [การเลือกพื้นที่ด้วยความสว่าง](#lesson-2)
3. [Threshold แต่ละประเภท](#lesson-3)
4. [การเลือกวัตถุด้วยสี](#lesson-4)
5. [การแปลงตำแหน่งและรูปทรง](#lesson-5)
6. [Affine Transformation](#lesson-6)
7. [Perspective Transformation](#lesson-7)
8. [การเลือกพื้นที่และปรับขนาดภาพ](#lesson-8)

[เฉลยแบบฝึกหัด](#solutions) · [เครดิตและการเรียนรู้เพิ่มเติม](#credits)

<a id="setup"></a>
## เตรียมข้อมูลอัตโนมัติ

รันเซลล์ถัดไปก่อนเริ่มบทเรียน ระบบเตรียมภาพครบ 16 ไฟล์ใน `dataset/` ตรวจ SHA-256 และทดลองเปิดด้วย OpenCV หากรันซ้ำจะใช้ไฟล์ที่ตรวจผ่านแล้ว รูปที่ขาดหรือเสียจะดาวน์โหลดใหม่

เมื่อเห็น **พร้อมเรียน: ตรวจภาพครบ 16 ไฟล์** ให้รันตัวอย่างหรือวางโค้ดจาก AI ในเซลล์ที่ต้องการ ทุก Prompt ใช้เส้นทาง `dataset/` เหมือนกัน ห้ามเปลี่ยน working directory ระหว่างบทเรียน

Colab เปิด notebook จาก GitHub โดยไม่ได้คัดลอกไฟล์ข้างเคียงเข้า runtime ให้เอง จึงต้องรันเซลล์นี้ก่อน ([Colab FAQ](https://research.google.com/colaboratory/faq.html)).

In [ ]:
# รันเซลล์นี้ก่อนทุกครั้งที่เริ่ม runtime ใหม่
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError, HTTPError
import hashlib
import importlib.util
import os
import subprocess
import sys
import time

# ติดตั้งเฉพาะไลบรารีที่ยังไม่มี (Colab ปกติมีครบแล้ว)
packages = {"cv2": "opencv-python-headless", "numpy": "numpy", "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import cv2
import numpy as np
import matplotlib.pyplot as plt

REPO = "bluebox-dev/Prompt-Computer-Vision"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/dataset/"
# SHA-256 ตรวจว่าข้อมูลตรงกับภาพที่ใช้สร้างบทเรียน
EXPECTED_SHA256 = {
    "butterfly.jpg": "34a99716b78ded10eae87b4880de739d3e68cc7dd1a96812bb28f469d5eeaea8",
    "cameraman.tif": "e2e6a7b862a5c1097e01775bee4bd537a1fc5ca03cef5e4d47f8f0bcc921acd0",
    "checkerboard.png": "36a04f9701b424550dbff7cf8c4e5cd7a12e68cea45a98774285af7799492254",
    "coins.jpg": "44c14af9d577ec99ac24687b26258a320d89cc974bf372d3b5340f42f545c71a",
    "color.bmp": "573a8fc6602a5ab82f5242910783a59c9b07e44c1e58ac660d40d959bc3b3835",
    "colorobject.png": "f7f8931f15d9bc7b9af9dd7df6996d9d3608c3b5be385658b9084eec1a3ecc09",
    "gray.bmp": "9a1f8ca1a966dfa2a88a2bee83f96211b1aee9a34a6e4b0d78c27429c404d9ce",
    "graylevel.jpg": "f068eb8cebbc0e077db338a0080d909ce769f40efbd5de1bc72c4132b050deeb",
    "left.jpg": "fb314330c3eb0a81651c682803e73b348898157267fbc37c4c2c07d1b3a8e321",
    "lena_color_256.tif": "93306889b7b31293b5b416dd2191600d1f1abf55049f282f914bf63a1fe38821",
    "messi.jpg": "e1c035ca5659c5e6560cd85db6bce36ff284f960eeb61cc6b455eebf4aa23cc5",
    "rectangle.png": "b6cd5b81b99999b1c504823e168b6cacef2d676d646c3ea5fcb90ac1187f02bf",
    "rectangle2.png": "345bf6a2eaf49ccbbbc8402acf544fbc0268e38eb0a58f450a62ea06e8682b02",
    "rectanglePoint.png": "012c5aa339194df6339e86185b9d216b40fae832e346023ba17361ab5940f1a9",
    "rectangleRotate.png": "c93f773512cb63aab060e9809e87a8d7104ea65f25a6692f7ffc6b3d94f637cf",
    "sudoku.jpg": "7f55352ab93cddff5950eb9b29f71053365b978ba4932513b8bcd3788a7055c7"
}

# Colab ใช้ /content เสมอ; Jupyter ใช้โฟลเดอร์ repo ปัจจุบันหรือ parent
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    WORK_DIR = Path("/content")
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    WORK_DIR = next((p for p in candidates if (p / "Computer_Vision_AI_Workshop.ipynb").is_file()), Path.cwd())
os.chdir(WORK_DIR)
DATA_DIR = WORK_DIR / "dataset"
DATA_DIR.mkdir(parents=True, exist_ok=True)

def valid_file(path, expected):
    return path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == expected

downloaded = 0
for filename, expected in EXPECTED_SHA256.items():
    destination = DATA_DIR / filename
    if not valid_file(destination, expected):
        for attempt in range(3):
            try:
                with urlopen(BASE_URL + filename, timeout=30) as response:
                    payload = response.read()
                if hashlib.sha256(payload).hexdigest() != expected:
                    raise ValueError(f"ข้อมูล {filename} ไม่ตรงกับรุ่นของ notebook; ให้ผู้สอนตรวจชุดไฟล์ที่เผยแพร่")
                temporary = destination.with_name(destination.name + ".download")
                temporary.write_bytes(payload)
                temporary.replace(destination)
                downloaded += 1
                break
            except HTTPError as exc:
                if exc.code in (403, 404):
                    raise RuntimeError(f"โหลด {filename} ไม่ได้ (HTTP {exc.code}): ตรวจว่า {REPO} เป็น public และมี dataset/ บน branch {BRANCH}") from exc
                if attempt == 2:
                    raise RuntimeError(f"โหลด {filename} ไม่สำเร็จ ให้รันเซลล์นี้ใหม่") from exc
                time.sleep(attempt + 1)
            except (URLError, TimeoutError) as exc:
                if attempt == 2:
                    raise RuntimeError(f"เชื่อมต่อ GitHub ไม่สำเร็จขณะโหลด {filename}; ตรวจอินเทอร์เน็ตแล้วรันเซลล์นี้ใหม่") from exc
                time.sleep(attempt + 1)
    image = cv2.imread(str(destination), cv2.IMREAD_UNCHANGED)
    if image is None:
        raise RuntimeError(f"OpenCV เปิดภาพไม่ได้: {destination}")

print(f"พร้อมเรียน: ตรวจภาพครบ {len(EXPECTED_SHA256)} ไฟล์ (ดาวน์โหลดใหม่ {downloaded} ไฟล์)")
print(f"โฟลเดอร์ทำงาน: {WORK_DIR}")
print(f"OpenCV {cv2.__version__} | NumPy {np.__version__}")
print("ตัวอย่างเส้นทาง: dataset/butterfly.jpg — รันตัวอย่างในบทเรียนต่อได้เลย")


<a id="lesson-1"></a>
## 1. ภาพดิจิทัลและระบบสี

**สไลด์ประกอบ:** [หน้า 14–15](#slide-1)

ภาพดิจิทัลประกอบด้วย **Pixel (พิกเซล)** ภาพเทา 8 บิตเก็บความสว่างหนึ่งค่าต่อจุด ตั้งแต่ 0 ถึง 255 ค่าน้อยมืด ค่ามากสว่าง

![พิกเซลและค่าความสว่างในภาพจริง](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/pixels.png)

ภาพสีเก็บสามช่อง OpenCV เรียงเป็น BGR การแยกแล้วรวมช่องกลับได้สีเดิม ส่วน Grayscale เฉลี่ยหรือถ่วงน้ำหนักสามช่องให้เหลือความสว่างค่าเดียว

![เปรียบเทียบภาพสี ภาพเทา และสีที่มีความสว่างใกล้กัน](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/concept_1.png)

สังเกตคู่สีทางขวา: แดงกับเขียวต่างกันชัด แต่เป็นเทาใกล้กัน **จึงควรเก็บสีไว้เมื่อใช้สีแยกวัตถุ**

<a id="example-1"></a>
### ตัวอย่างที่ 1: แยกช่องสีและแปลงเป็น Grayscale

**วัตถุประสงค์:** เห็นว่าการเปลี่ยนเป็นภาพเทาสูญเสียข้อมูลสีอะไร

**Prompt ตัวอย่าง**

```text
Python/Google Colab ใช้ OpenCV, NumPy, Matplotlib อ่าน dataset/butterfly.jpg
แสดงต้นฉบับ ช่อง B, G, R และ Grayscale แบบเฉลี่ยกับแบบถ่วงน้ำหนัก
รวม BGR กลับแล้วพิมพ์ว่าเท่ากับต้นฉบับหรือไม่
ตอบเฉพาะโค้ด 1 เซลล์ ไม่เกิน 25 บรรทัด ใช้คำสั่งพื้นฐาน ไม่สร้างฟังก์ชันหรือคลาส
หนึ่งคำสั่งต่อบรรทัด ไม่เพิ่มโค้ดติดตั้ง ค้นหาไฟล์ หรือคำอธิบายนอกโค้ด
```

**ผลที่ควรสังเกต:** มีภาพเทาให้เทียบกับภาพสี และรวมช่องกลับได้สีเดิม

[ดูภาพผลลัพธ์ของตัวอย่างที่ 1](#example-result-1)

In [ ]:
# ตัวอย่างที่ 1: พร้อมรัน หรือแทนด้วยโค้ดจาก Prompt ของคุณ
import cv2
import numpy as np
import matplotlib.pyplot as plt
img = cv2.imread("dataset/butterfly.jpg")
b, g, r = cv2.split(img)
average = np.mean(img.astype(np.float32), axis=2).astype(np.uint8)
weighted = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
merged = cv2.merge([b, g, r])
print("Merged BGR equals original:", np.array_equal(img, merged))
images = [cv2.cvtColor(img, cv2.COLOR_BGR2RGB), b, g, r, average, weighted]
titles = ["Original RGB", "Blue channel", "Green channel", "Red channel", "Mean grayscale", "Weighted grayscale"]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, picture, title in zip(axes.flat, images, titles):
    ax.imshow(picture, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


<a id="exercise-1"></a>
### แบบฝึกหัดที่ 1: ปรับโทนสีภาพ

**โจทย์:** เลือกปรับภาพผีเสื้อเป็นโทนอุ่นหรือโทนเย็น โดยคงขนาดและรูปทรงเดิม

**ประเด็นให้คิด:** ช่องสีใดควรเปลี่ยน และควรเปลี่ยนมากเพียงใด?

**Prompt ตั้งต้น**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน butterfly.jpg จาก dataset/
ปรับภาพเป็นโทน [อุ่นหรือเย็น] ระดับ [อ่อนหรือชัด]
ใช้การปรับช่องสี คงขนาดและรูปทรงเดิม แสดง Before/After และบอกว่าปรับช่องใด
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

**เกณฑ์ผลงาน:** โทนภาพเปลี่ยนตามที่ระบุ ขนาดและรูปทรงเดิมไม่เปลี่ยน

[ดูเฉลยแบบฝึกหัดที่ 1](#solution-1)

**Prompt ที่ใช้และเหตุผลที่เลือก**

บันทึกคำขอสุดท้ายและเหตุผลสั้น ๆ ของคุณที่นี่

In [ ]:
# แบบฝึกหัดที่ 1
# วางโค้ดที่ได้รับจาก AI แล้วรันในเซลล์นี้

<a id="lesson-2"></a>
## 2. การเลือกพื้นที่ด้วยความสว่าง

**สไลด์ประกอบ:** [หน้า 18–23](#slide-2)

**Histogram (กราฟความสว่าง)** นับจำนวนพิกเซลในแต่ละระดับความสว่าง ยอดกราฟบอกว่าค่าใดมีมาก ส่วนเส้นสีส้มคือ Threshold หรือค่าที่ใช้แบ่ง

![ความสัมพันธ์ระหว่างภาพ Histogram และ Mask](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/concept_2.png)

ดูภาพขวา: Binary เปลี่ยนค่าที่มากกว่า 100 เป็นขาว ค่าที่ไม่เกิน 100 เป็นดำ เช่น 88 ไม่ผ่าน แต่ 195 ผ่าน

ผลนี้คือ **Mask** สำหรับบอกพื้นที่เลือก ไม่ได้แปลว่าโปรแกรมรู้จักวัตถุแล้ว หากแสงเปลี่ยน ค่าแบ่งอาจต้องเปลี่ยนตาม

<a id="example-2"></a>
### ตัวอย่างที่ 2: เปรียบเทียบค่า Threshold

**วัตถุประสงค์:** เลือกพื้นที่จากความสว่าง

**Prompt ตัวอย่าง**

```text
Python/Google Colab ใช้ OpenCV, NumPy, Matplotlib อ่าน dataset/graylevel.jpg
อ่านเป็น Grayscale แสดงต้นฉบับ Histogram และ Binary Mask ที่ Threshold = 50, 100, 210
ตอบเฉพาะโค้ด 1 เซลล์ ไม่เกิน 25 บรรทัด ใช้คำสั่งพื้นฐาน ไม่สร้างฟังก์ชันหรือคลาส
หนึ่งคำสั่งต่อบรรทัด ไม่เพิ่มโค้ดติดตั้ง ค้นหาไฟล์ หรือคำอธิบายนอกโค้ด
```

**ผลที่ควรสังเกต:** จำนวนแท่งขาวลดจาก 3 เป็น 2 และ 1

[ดูภาพผลลัพธ์ของตัวอย่างที่ 2](#example-result-2)

In [ ]:
# ตัวอย่างที่ 2: พร้อมรัน หรือแทนด้วยโค้ดจาก Prompt ของคุณ
import cv2
import matplotlib.pyplot as plt
img = cv2.imread("dataset/graylevel.jpg", cv2.IMREAD_GRAYSCALE)
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
axes[0].imshow(img, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].hist(img.ravel(), bins=256, range=(0, 256))
axes[1].set_title("Histogram")
for ax, threshold in zip(axes[2:], [50, 100, 210]):
    _, mask = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY)
    ax.imshow(mask, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"Threshold = {threshold}")
    ax.axis("off")
plt.tight_layout()
plt.show()


<a id="exercise-2"></a>
### แบบฝึกหัดที่ 2: เลือกเฉพาะแท่งกลาง

**โจทย์:** สร้าง Mask ที่เลือกเฉพาะแท่งกลางเป็นสีขาว โดยไม่เลือกแท่งบน แท่งล่าง หรือพื้นหลัง

**ประเด็นให้คิด:** เหตุใดการกำหนดค่าแบ่งเพียงด้านเดียวอาจเลือกแท่งอื่นติดมาด้วย?

**Prompt ตั้งต้น**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน graylevel.jpg จาก dataset/
สร้าง Mask ที่เหลือเฉพาะแท่งกลางเป็นสีขาว และให้พื้นที่อื่นเป็นสีดำ
เลือกจากความสว่างของภาพจริง แสดงช่วงค่าที่ใช้และผลเทียบต้นฉบับ
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

**เกณฑ์ผลงาน:** Mask เหลือแท่งกลางเพียงแท่งเดียว

[ดูเฉลยแบบฝึกหัดที่ 2](#solution-2)

**Prompt ที่ใช้และเหตุผลที่เลือก**

บันทึกคำขอสุดท้ายและเหตุผลสั้น ๆ ของคุณที่นี่

In [ ]:
# แบบฝึกหัดที่ 2
# วางโค้ดที่ได้รับจาก AI แล้วรันในเซลล์นี้

<a id="lesson-3"></a>
## 3. Threshold แต่ละประเภท

**สไลด์ประกอบ:** [หน้า 20–28](#slide-3)

แถบแรกไล่จากมืดไปสว่าง แถบอื่นใช้ข้อมูลเดียวกันและแบ่งที่ 100 แต่ให้ค่าพิกเซลใหม่ต่างกัน

![แถบระดับเทาแสดงผล Threshold ห้าวิธี](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/concept_3.png)

**Binary** เลือกด้านสว่างเป็นขาว ส่วน **Binary inverse** เลือกด้านมืดเป็นขาว ทั้งคู่จึงเหมาะกับ Mask

**Trunc** ลดค่าที่เกิน 100 ให้เหลือ 100 ส่วน **To zero** ตัดด้านมืดและเก็บค่าด้านสว่างไว้ **To zero inverse** ทำกลับกัน ผลเหล่านี้ยังมีระดับเทาได้

<a id="example-3"></a>
### ตัวอย่างที่ 3: เปรียบเทียบ Threshold 5 ประเภท

**วัตถุประสงค์:** แยกการเลือกพื้นที่ออกจากการปรับระดับเทา

**Prompt ตัวอย่าง**

```text
Python/Google Colab ใช้ OpenCV, NumPy, Matplotlib อ่าน dataset/graylevel.jpg
อ่านเป็น Grayscale ใช้ Threshold = 100
แสดงต้นฉบับเทียบ Binary, Binary inverse, Trunc, To zero, To zero inverse พร้อมชื่อ
ตอบเฉพาะโค้ด 1 เซลล์ ไม่เกิน 25 บรรทัด ใช้คำสั่งพื้นฐาน ไม่สร้างฟังก์ชันหรือคลาส
หนึ่งคำสั่งต่อบรรทัด ไม่เพิ่มโค้ดติดตั้ง ค้นหาไฟล์ หรือคำอธิบายนอกโค้ด
```

**ผลที่ควรสังเกต:** Binary มีขาวกับดำ ส่วนวิธีอื่นยังอาจมีระดับเทา

[ดูภาพผลลัพธ์ของตัวอย่างที่ 3](#example-result-3)

In [ ]:
# ตัวอย่างที่ 3: พร้อมรัน หรือแทนด้วยโค้ดจาก Prompt ของคุณ
import cv2
import matplotlib.pyplot as plt
img = cv2.imread("dataset/graylevel.jpg", cv2.IMREAD_GRAYSCALE)
modes = [cv2.THRESH_BINARY, cv2.THRESH_BINARY_INV, cv2.THRESH_TRUNC, cv2.THRESH_TOZERO, cv2.THRESH_TOZERO_INV]
images = [img] + [cv2.threshold(img, 100, 255, mode)[1] for mode in modes]
titles = ["Original", "Binary", "Binary inverse", "Trunc", "To zero", "To zero inverse"]
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, picture, title in zip(axes.flat, images, titles):
    ax.imshow(picture, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


<a id="exercise-3"></a>
### แบบฝึกหัดที่ 3: รักษาระดับเทาของส่วนที่เลือก

**โจทย์:** เก็บเฉพาะแท่งมืดด้านบนให้มีระดับเทาเดิม และเปลี่ยนส่วนอื่นเป็นสีดำ

**ประเด็นให้คิด:** วิธีใดเก็บค่าพิกเซลเดิม แทนการเปลี่ยนพื้นที่ที่เลือกให้เป็นสีขาว?

**Prompt ตั้งต้น**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน graylevel.jpg จาก dataset/
เก็บแท่งมืดด้านบนด้วยระดับเทาเดิม
เปลี่ยนแท่งกลางและแท่งล่างเป็นสีดำ โดยไม่เปลี่ยนแท่งที่เก็บให้เป็นสีขาว
เลือกประเภท Threshold ที่เหมาะสมและแสดง Before/After
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

**เกณฑ์ผลงาน:** แท่งบนยังเป็นระดับเทาเดิม อีก 2 แท่งหายไป

[ดูเฉลยแบบฝึกหัดที่ 3](#solution-3)

**Prompt ที่ใช้และเหตุผลที่เลือก**

บันทึกคำขอสุดท้ายและเหตุผลสั้น ๆ ของคุณที่นี่

In [ ]:
# แบบฝึกหัดที่ 3
# วางโค้ดที่ได้รับจาก AI แล้วรันในเซลล์นี้

<a id="lesson-4"></a>
## 4. การเลือกวัตถุด้วยสี

**สไลด์ประกอบ:** [หน้า 29–42](#slide-4)

**HSV** อธิบายสีด้วย Hue (ชนิดสี), Saturation (ความอิ่มสี) และ Value (ความสว่าง) ดูแต่ละแถบซึ่งเปลี่ยนเพียงองค์ประกอบเดียว

![แถบสีอธิบาย Hue Saturation และ Value](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/hsv.png)

เลือกพิกเซลที่ H, S และ V อยู่ในช่วงที่ต้องการพร้อมกัน สีแดงอยู่ใกล้รอยต่อของ H จึงอาจต้องเลือกสองช่วงแล้วรวมกัน

![การใช้เงื่อนไขสีสร้าง Mask และเก็บภาพเดิม](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/concept_4.png)

ดูภาพกลาง: ขาวคือพื้นที่น้ำเงินที่เลือก ภาพขวาใช้พื้นที่นั้นเก็บสีเดิม ถ้าต้องการแดงหรือน้ำเงิน ให้รวม Mask แบบ **OR** แล้วนำไปใช้งานต่อ

<a id="example-4"></a>
### ตัวอย่างที่ 4: เลือกสีและเปลี่ยนสีเฉพาะพื้นที่

**วัตถุประสงค์:** เลือกสีและนำพื้นที่นั้นไปสร้างผลลัพธ์

**Prompt ตัวอย่าง**

```text
Python/Google Colab ใช้ OpenCV, NumPy, Matplotlib อ่าน dataset/colorobject.png
วางพื้นโปร่งใสบนสีขาว ใช้ HSV เลือกแดงกับน้ำเงิน
แสดงต้นฉบับ Mask ภาพเก็บสีบนพื้นดำ และภาพเปลี่ยนสีที่เลือกเป็นเขียว ส่วนอื่นคงเดิม
ตอบเฉพาะโค้ด 1 เซลล์ ไม่เกิน 35 บรรทัด ใช้คำสั่งพื้นฐาน ไม่สร้างฟังก์ชันหรือคลาส
หนึ่งคำสั่งต่อบรรทัด ไม่เพิ่มโค้ดติดตั้ง ค้นหาไฟล์ หรือคำอธิบายนอกโค้ด
```

**ผลที่ควรสังเกต:** แดงกับน้ำเงินรวม 6 ชิ้นถูกเลือก พื้นหลังไม่ติด Mask

[ดูภาพผลลัพธ์ของตัวอย่างที่ 4](#example-result-4)

In [ ]:
# ตัวอย่างที่ 4: พร้อมรัน หรือแทนด้วยโค้ดจาก Prompt ของคุณ
import cv2
import numpy as np
import matplotlib.pyplot as plt
rgba = cv2.imread("dataset/colorobject.png", cv2.IMREAD_UNCHANGED)
alpha = rgba[:, :, 3:4].astype(np.float32) / 255
img = np.round(rgba[:, :, :3] * alpha + 255 * (1 - alpha)).astype(np.uint8)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
red1 = cv2.inRange(hsv, (0, 80, 50), (10, 255, 255))
red2 = cv2.inRange(hsv, (170, 80, 50), (179, 255, 255))
blue = cv2.inRange(hsv, (100, 80, 50), (130, 255, 255))
mask = cv2.bitwise_or(cv2.bitwise_or(red1, red2), blue)
selected = cv2.bitwise_and(img, img, mask=mask)
recolored = img.copy()
recolored[mask > 0] = (0, 255, 0)
images = [cv2.cvtColor(img, cv2.COLOR_BGR2RGB), mask, cv2.cvtColor(selected, cv2.COLOR_BGR2RGB), cv2.cvtColor(recolored, cv2.COLOR_BGR2RGB)]
titles = ["Original on white", "Red OR blue mask", "Selected colors", "Recolored green"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, picture, title in zip(axes.flat, images, titles):
    ax.imshow(picture, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


<a id="exercise-4"></a>
### แบบฝึกหัดที่ 4: กำหนดสีเป้าหมายและสีเน้น

**โจทย์:** เลือกวัตถุ 2 สี แล้วเปลี่ยนพื้นที่ที่เลือกเป็นสีเน้นที่กำหนดเอง โดยคงพื้นที่อื่นเดิม

**ประเด็นให้คิด:** จะเลือก 2 สีใด และจะตรวจว่ามีสีอื่นติด Mask มาหรือไม่?

**Prompt ตั้งต้น**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน colorobject.png จาก dataset/
วางพื้นที่โปร่งใสบนพื้นสีขาว
ใช้ HSV เลือกวัตถุสี [สีที่ 1] และ [สีที่ 2] แล้วเปลี่ยนพื้นที่นั้นเป็น [สีเน้น]
คงสีอื่น พื้นหลังและขนาดภาพเดิม แสดงต้นฉบับ Mask และภาพผลลัพธ์
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

**เกณฑ์ผลงาน:** เลือกตรงกับ 2 สีที่ระบุ และเปลี่ยนสีเฉพาะพื้นที่นั้น

[ดูเฉลยแบบฝึกหัดที่ 4](#solution-4)

**Prompt ที่ใช้และเหตุผลที่เลือก**

บันทึกคำขอสุดท้ายและเหตุผลสั้น ๆ ของคุณที่นี่

In [ ]:
# แบบฝึกหัดที่ 4
# วางโค้ดที่ได้รับจาก AI แล้วรันในเซลล์นี้

<a id="lesson-5"></a>
## 5. การแปลงตำแหน่งและรูปทรง

**สไลด์ประกอบ:** [หน้า 53–60](#slide-5)

พิกัดภาพเริ่มมุมซ้ายบน **x เพิ่มทางขวา ส่วน y เพิ่มลงล่าง** จุดสีส้มอยู่ห่างจากซ้าย 200 และจากบน 100 พิกเซล

![ระบบพิกัดภาพและตำแหน่งตัวอย่าง](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/coordinates.png)

Translation เปลี่ยนตำแหน่ง Scaling เปลี่ยนขนาด Rotation เปลี่ยนทิศทาง และ Shear เปลี่ยนความเอียง ดูผลกับภาพเดียวกันด้านล่าง

![ภาพเปรียบเทียบการเลื่อน ย่อ หมุน และ Shear](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/concept_5.png)

สังเกตภาพย่อ: วัตถุเล็กลงแต่กรอบยังเท่าเดิม **ต้องกำหนดทั้งการแปลงและขนาดกรอบ** หากเนื้อภาพพ้นกรอบจะถูกตัด การสลับลำดับการแปลงอาจให้ผลต่างกัน

<a id="example-5"></a>
### ตัวอย่างที่ 5: เลื่อน ย่อ หมุน และ Shear

**วัตถุประสงค์:** จัดตำแหน่งโดยแยกวัตถุออกจากกรอบภาพ

**Prompt ตัวอย่าง**

```text
Python/Google Colab ใช้ OpenCV, NumPy, Matplotlib อ่าน dataset/rectangle.png
แปลงจากต้นฉบับแยกกัน: เลื่อนขวา 100 ลง 50 พิกเซล; ย่อ 0.5 เท่ารอบกลางภาพ; หมุนทวนเข็ม 25.5 องศา
ใช้กรอบขาวกว้าง 760 สูง 500 พิกเซล ส่วน Shear แนว X = 0.3 ให้ขยายกรอบจนวัตถุอยู่ครบ แสดงเทียบต้นฉบับ
ตอบเฉพาะโค้ด 1 เซลล์ ไม่เกิน 35 บรรทัด ใช้คำสั่งพื้นฐาน ไม่สร้างฟังก์ชันหรือคลาส
หนึ่งคำสั่งต่อบรรทัด ไม่เพิ่มโค้ดติดตั้ง ค้นหาไฟล์ หรือคำอธิบายนอกโค้ด
```

**ผลที่ควรสังเกต:** เห็นความต่างของการเลื่อน ย่อ หมุนและเอียง

[ดูภาพผลลัพธ์ของตัวอย่างที่ 5](#example-result-5)

In [ ]:
# ตัวอย่างที่ 5: พร้อมรัน หรือแทนด้วยโค้ดจาก Prompt ของคุณ
import cv2
import numpy as np
import matplotlib.pyplot as plt
rgba = cv2.imread("dataset/rectangle.png", cv2.IMREAD_UNCHANGED)
alpha = rgba[:, :, 3:4].astype(np.float32) / 255
img = np.round(rgba[:, :, :3] * alpha + 255 * (1 - alpha)).astype(np.uint8)
h, w = img.shape[:2]
translation = np.float32([[1, 0, 100], [0, 1, 50]])
scaling = cv2.getRotationMatrix2D((w / 2, h / 2), 0, 0.5)
rotation = cv2.getRotationMatrix2D((w / 2, h / 2), 25.5, 1)
shear = np.float32([[1, 0.3, 0], [0, 1, 0]])
matrices = [translation, scaling, rotation, shear]
sizes = [(w, h), (w, h), (w, h), (w + int(np.ceil(0.3 * h)), h)]
images = [img] + [cv2.warpAffine(img, matrix, size, borderValue=(255, 255, 255)) for matrix, size in zip(matrices, sizes)]
titles = ["Original", "Translate (100, 50)", "Scale 0.5 around center", "Rotate 25.5 degrees", "Shear X = 0.3"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, picture, title in zip(axes.flat, images, titles):
    ax.imshow(cv2.cvtColor(picture, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()


<a id="exercise-5"></a>
### แบบฝึกหัดที่ 5: จัดวางวัตถุในกรอบภาพ

**โจทย์:** ย่อวัตถุและจัดวางให้เหลือพื้นที่ว่างด้านหนึ่ง โดยคงกรอบภาพเดิมและไม่ตัดวัตถุ

**ประเด็นให้คิด:** จะย่อเท่าไร วางด้านใด และควรทำขั้นตอนไหนก่อน?

**Prompt ตั้งต้น**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน rectangle.png จาก dataset/
ย่อเนื้อภาพเป็น [อัตราย่อ เช่น 0.5] เท่า แล้วจัดวางทาง [ซ้าย กลาง หรือขวา]
คงกรอบสีขาวกว้าง 760 สูง 500 พิกเซล เก็บวัตถุครบและแสดง Before/After
ระบุค่าการเลื่อนที่ใช้
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

**เกณฑ์ผลงาน:** วัตถุเล็กลง อยู่ในตำแหน่งที่เลือก และกรอบยังเป็น 760 คูณ 500 พิกเซล

[ดูเฉลยแบบฝึกหัดที่ 5](#solution-5)

**Prompt ที่ใช้และเหตุผลที่เลือก**

บันทึกคำขอสุดท้ายและเหตุผลสั้น ๆ ของคุณที่นี่

In [ ]:
# แบบฝึกหัดที่ 5
# วางโค้ดที่ได้รับจาก AI แล้วรันในเซลล์นี้

<a id="lesson-6"></a>
## 6. Affine Transformation

**สไลด์ประกอบ:** [หน้า 61–62](#slide-6)

**Affine** ใช้จุดต้นทางสามจุดจับคู่กับจุดปลายทางสามจุด จุดสีเดียวกันในภาพเป็นคู่เดียวกัน

![สามคู่จุดกำหนดการแปลง Affine](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/concept_6.png)

กติกาจากสามคู่นี้เปลี่ยนทั้งภาพ โดยแต่ละชุดต้องไม่อยู่บนเส้นตรงเดียวกัน Affine รวมการเลื่อน หมุน ย่อขยาย และ Shear ได้ เส้นขนานยังขนาน แต่มุมและความยาวเปลี่ยนได้

สังเกตรูปขวา: ต้องปรับความเอียงและสัดส่วนจึงเป็นจัตุรัส **การหมุนอย่างเดียวรักษามุมภายในรูป จึงแก้โจทย์นี้ไม่พอ**

<a id="example-6"></a>
### ตัวอย่างที่ 6: เปลี่ยนสี่เหลี่ยมด้านขนานเป็นจัตุรัส

**วัตถุประสงค์:** กำหนดรูปทรงด้วยจุดอ้างอิง 3 คู่

**Prompt ตัวอย่าง**

```text
Python/Google Colab ใช้ OpenCV, NumPy, Matplotlib อ่าน dataset/rectangle2.png
ใช้ Affine จากจุด (243,110), (682,110), (80,382) ไป (255,125), (505,125), (255,375) ตามลำดับ
ใช้กรอบขาวกว้าง 760 สูง 500 พิกเซล แสดงต้นฉบับพร้อมจุด และผลลัพธ์
ตอบเฉพาะโค้ด 1 เซลล์ ไม่เกิน 30 บรรทัด ใช้คำสั่งพื้นฐาน ไม่สร้างฟังก์ชันหรือคลาส
หนึ่งคำสั่งต่อบรรทัด ไม่เพิ่มโค้ดติดตั้ง ค้นหาไฟล์ หรือคำอธิบายนอกโค้ด
```

**ผลที่ควรสังเกต:** ได้จัตุรัสกลางภาพ ไม่ใช่รูปด้านขนานที่หมุนแล้ว

[ดูภาพผลลัพธ์ของตัวอย่างที่ 6](#example-result-6)

In [ ]:
# ตัวอย่างที่ 6: พร้อมรัน หรือแทนด้วยโค้ดจาก Prompt ของคุณ
import cv2
import numpy as np
import matplotlib.pyplot as plt
rgba = cv2.imread("dataset/rectangle2.png", cv2.IMREAD_UNCHANGED)
alpha = rgba[:, :, 3:4].astype(np.float32) / 255
img = np.round(rgba[:, :, :3] * alpha + 255 * (1 - alpha)).astype(np.uint8)
src = np.float32([[243, 110], [682, 110], [80, 382]])
dst = np.float32([[255, 125], [505, 125], [255, 375]])
matrix = cv2.getAffineTransform(src, dst)
result = cv2.warpAffine(img, matrix, (760, 500), borderValue=(255, 255, 255))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, picture, points, title in zip(axes, [img, result], [src, dst], ["Source points", "Affine: 250 x 250"]):
    ax.imshow(cv2.cvtColor(picture, cv2.COLOR_BGR2RGB))
    ax.scatter(points[:, 0], points[:, 1], c=["red", "orange", "lime"], s=50)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


<a id="exercise-6"></a>
### แบบฝึกหัดที่ 6: ออกแบบป้ายสี่เหลี่ยม

**โจทย์:** ทำรูปสีน้ำเงินเป็นป้ายสี่เหลี่ยมผืนผ้ากลางภาพ โดยให้ความกว้างเป็น 2 เท่าของความสูง

**ประเด็นให้คิด:** จุดปลายทางต้องวางอย่างไรจึงได้ทั้งมุมฉากและสัดส่วนที่ต้องการ?

**Prompt ตั้งต้น**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน rectangle2.png จาก dataset/
ใช้ Affine Transformation เปลี่ยนรูปสีน้ำเงินเป็นป้ายกว้าง [ความกว้าง] สูง [ความสูง] พิกเซล
ให้กว้างเป็น 2 เท่าของสูง และอยู่กลางกรอบสีขาวกว้าง 760 สูง 500 พิกเซล
แสดงจุดต้นทาง จุดปลายทาง และผลที่ไม่มีเส้นช่วย
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

**เกณฑ์ผลงาน:** ป้ายมีมุมฉาก กว้างประมาณ 2 เท่าของสูง และอยู่กลางภาพ

[ดูเฉลยแบบฝึกหัดที่ 6](#solution-6)

**Prompt ที่ใช้และเหตุผลที่เลือก**

บันทึกคำขอสุดท้ายและเหตุผลสั้น ๆ ของคุณที่นี่

In [ ]:
# แบบฝึกหัดที่ 6
# วางโค้ดที่ได้รับจาก AI แล้วรันในเซลล์นี้

<a id="lesson-7"></a>
## 7. Perspective Transformation

**สไลด์ประกอบ:** [หน้า 63–64](#slide-7)

มุมกล้องทำให้ขอบปกที่ขนานกันจริงดูเหมือนลู่เข้า **Perspective Transform** แก้ปัญหานี้ด้วยสี่คู่จุดบนพื้นผิวระนาบ

![สี่มุมปกจับคู่กับสี่มุมภาพปลายทาง](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/concept_7.png)

ดูจุดสีเดียวกัน: แต่ละมุมปกไปอยู่มุมเดียวกันในภาพใหม่ ต้องเรียงลำดับให้ตรงกันจึงไม่บิดหรือกลับด้าน ต่างจาก Affine ที่ใช้สามคู่และรักษาความขนาน

**เลือกมุมให้ถูกและกำหนดสัดส่วนให้เหมาะ** วิธีนี้ไม่กู้รายละเอียดที่หายไป และไม่แก้หน้ากระดาษโค้งทั้งหมด

<a id="example-7"></a>
### ตัวอย่างที่ 7: แก้มุมมองปกหนังสือ

**วัตถุประสงค์:** แก้มุมกล้องด้วยจุดอ้างอิง 4 คู่

**Prompt ตัวอย่าง**

```text
Python/Google Colab ใช้ OpenCV, NumPy, Matplotlib อ่าน dataset/left.jpg
ใช้ Perspective แก้ปกให้กว้าง 300 สูง 420 พิกเซล
มุมซ้ายบน ขวาบน ขวาล่าง ซ้ายล่าง: (280,110), (394,111), (402,292), (298,325)
แสดงต้นฉบับพร้อมจุด และปกที่แก้แล้วโดยไม่กลับด้านข้อความ
ตอบเฉพาะโค้ด 1 เซลล์ ไม่เกิน 30 บรรทัด ใช้คำสั่งพื้นฐาน ไม่สร้างฟังก์ชันหรือคลาส
หนึ่งคำสั่งต่อบรรทัด ไม่เพิ่มโค้ดติดตั้ง ค้นหาไฟล์ หรือคำอธิบายนอกโค้ด
```

**ผลที่ควรสังเกต:** ปกเต็มกรอบและข้อความอ่านในทิศทางเดิม

[ดูภาพผลลัพธ์ของตัวอย่างที่ 7](#example-result-7)

In [ ]:
# ตัวอย่างที่ 7: พร้อมรัน หรือแทนด้วยโค้ดจาก Prompt ของคุณ
import cv2
import numpy as np
import matplotlib.pyplot as plt
img = cv2.imread("dataset/left.jpg")
src = np.float32([[280, 110], [394, 111], [402, 292], [298, 325]])
dst = np.float32([[0, 0], [299, 0], [299, 419], [0, 419]])
matrix = cv2.getPerspectiveTransform(src, dst)
result = cv2.warpPerspective(img, matrix, (300, 420))
fig, axes = plt.subplots(1, 2, figsize=(10, 6))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].scatter(src[:, 0], src[:, 1], c=["red", "orange", "lime", "cyan"], s=40)
axes[0].set_title("Source: TL, TR, BR, BL")
axes[1].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
axes[1].set_title("Perspective: 300 x 420")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


<a id="exercise-7"></a>
### แบบฝึกหัดที่ 7: จัดทำภาพปกพร้อมขอบ

**โจทย์:** แก้มุมมองปกหนังสือแล้วเพิ่มขอบรอบภาพ โดยเลือกสีและความกว้างของขอบเอง

**ประเด็นให้คิด:** ต้องทำให้ปกตรงก่อนหรือเพิ่มขอบก่อน และขนาดภาพสุดท้ายจะเปลี่ยนอย่างไร?

**Prompt ตั้งต้น**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน left.jpg จาก dataset/
แก้มุมมองปก BASIC ELECTRONICS ให้กว้าง 300 สูง 420 พิกเซล
ใช้จุด (280,110), (394,111), (402,292), (298,325) เรียงซ้ายบน ขวาบน ขวาล่าง ซ้ายล่าง
เพิ่มขอบสี [สีขอบ] หนา [จำนวน] พิกเซลทุกด้าน ไม่ยืดปก และรายงานขนาดภาพสุดท้าย
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

**เกณฑ์ผลงาน:** ปกไม่บิดหรือกลับด้าน ขอบตรงตามที่ระบุ และขนาดภาพเพิ่มตามความกว้างขอบ

[ดูเฉลยแบบฝึกหัดที่ 7](#solution-7)

**Prompt ที่ใช้และเหตุผลที่เลือก**

บันทึกคำขอสุดท้ายและเหตุผลสั้น ๆ ของคุณที่นี่

In [ ]:
# แบบฝึกหัดที่ 7
# วางโค้ดที่ได้รับจาก AI แล้วรันในเซลล์นี้

<a id="lesson-8"></a>
## 8. การเลือกพื้นที่และปรับขนาดภาพ

**สไลด์ประกอบ:** [หน้า 67–72](#slide-8)

**Crop** เลือกว่าจะเก็บส่วนไหน **Resize** กำหนดขนาดส่งออก ตัวอย่างนี้ใช้กรอบครอปกว้างเพื่อให้เห็นผลของสัดส่วนชัดเจน

![เปรียบเทียบการยืดภาพกับการเติมขอบก่อน Resize](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/concept_8.png)

ดูภาพกลาง: บังคับกรอบกว้างให้เป็นจัตุรัสทำให้บอลยืด ภาพขวาเติมขอบก่อนจึงรักษารูปทรงเดิม

การ Resize ต้องประมาณค่าพิกเซลใหม่ มักใช้ Area เมื่อลดขนาด และ Linear หรือ Cubic เมื่อขยาย **จำนวนพิกเซลเพิ่มได้ แต่รายละเอียดจริงไม่ได้เพิ่มตาม**

<a id="example-8"></a>
### ตัวอย่างที่ 8: Crop ลูกบอลและ Resize โดยคงสัดส่วน

**วัตถุประสงค์:** เลือกส่วนที่ต้องการและรักษาสัดส่วน

**Prompt ตัวอย่าง**

```text
Python/Google Colab ใช้ OpenCV, NumPy, Matplotlib อ่าน dataset/messi.jpg
Crop บอลที่ x=330:395, y=280:342 (ไม่รวมค่าปลาย)
เติมขอบขาวเป็นจัตุรัส แล้ว Resize เป็น 300 × 300 พิกเซล
แสดงกรอบบนต้นฉบับ ภาพ Crop และผลลัพธ์
ตอบเฉพาะโค้ด 1 เซลล์ ไม่เกิน 25 บรรทัด ใช้คำสั่งพื้นฐาน ไม่สร้างฟังก์ชันหรือคลาส
หนึ่งคำสั่งต่อบรรทัด ไม่เพิ่มโค้ดติดตั้ง ค้นหาไฟล์ หรือคำอธิบายนอกโค้ด
```

**ผลที่ควรสังเกต:** บอลอยู่ครบและไม่ยืดเป็นวงรี

[ดูภาพผลลัพธ์ของตัวอย่างที่ 8](#example-result-8)

In [ ]:
# ตัวอย่างที่ 8: พร้อมรัน หรือแทนด้วยโค้ดจาก Prompt ของคุณ
import cv2
import matplotlib.pyplot as plt
img = cv2.imread("dataset/messi.jpg")
crop = img[280:342, 330:395]
h, w = crop.shape[:2]
side = max(h, w)
top = (side - h) // 2
left = (side - w) // 2
square = cv2.copyMakeBorder(crop, top, side - h - top, left, side - w - left, cv2.BORDER_CONSTANT, value=(255, 255, 255))
result = cv2.resize(square, (300, 300), interpolation=cv2.INTER_CUBIC)
marked = img.copy()
cv2.rectangle(marked, (330, 280), (394, 341), (0, 255, 0), 2)
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, picture, title in zip(axes, [marked, crop, result], ["Crop location", "Crop 65 x 62", "Padded and resized 300 x 300"]):
    ax.imshow(cv2.cvtColor(picture, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


<a id="exercise-8"></a>
### แบบฝึกหัดที่ 8: เตรียมภาพบอลแนวนอน

**โจทย์:** เตรียมภาพลูกบอลกว้าง 400 สูง 300 พิกเซล โดยเก็บลูกบอลครบและไม่ยืดรูปทรง

**ประเด็นให้คิด:** ควรปรับขนาด เติมขอบ หรือ Crop เพิ่มอย่างไรเพื่อให้ตรงสัดส่วนปลายทาง?

**Prompt ตั้งต้น**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน messi.jpg จาก dataset/
Crop ลูกบอลด้วยกรอบ x ตั้งแต่ 330 ถึงก่อน 395 และ y ตั้งแต่ 280 ถึงก่อน 342
ทำภาพสุดท้ายกว้าง 400 สูง 300 พิกเซล เก็บลูกบอลครบและคงรูปทรง
เลือกวิธีรักษาสัดส่วนที่เหมาะสม แสดงกรอบ Crop และผลสุดท้าย
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

**เกณฑ์ผลงาน:** ผลลัพธ์กว้าง 400 สูง 300 พิกเซล ลูกบอลอยู่ครบและไม่ยืด

[ดูเฉลยแบบฝึกหัดที่ 8](#solution-8)

**Prompt ที่ใช้และเหตุผลที่เลือก**

บันทึกคำขอสุดท้ายและเหตุผลสั้น ๆ ของคุณที่นี่

In [ ]:
# แบบฝึกหัดที่ 8
# วางโค้ดที่ได้รับจาก AI แล้วรันในเซลล์นี้

<a id="solutions"></a>
# ผลลัพธ์ตัวอย่างและเฉลยแบบฝึกหัด

ภาพผลลัพธ์ด้านล่างประมวลจาก dataset ส่วนเฉลยแบบฝึกหัดเป็นแนวทางหนึ่ง หากเลือกค่าหรือรูปแบบอื่น ให้ตรวจตามเกณฑ์ของโจทย์

<a id="example-result-1"></a>
## ผลลัพธ์ตัวอย่างที่ 1: แยกช่องสีและแปลงเป็น Grayscale

มีภาพเทาให้เทียบกับภาพสี และรวมช่องกลับได้สีเดิม

![ตัวอย่างผลลัพธ์ข้อ 1](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/answer_1_0.png)

[กลับสู่ตัวอย่างที่ 1](#example-1)

<a id="solution-1"></a>
## เฉลยแบบฝึกหัดที่ 1: ปรับโทนสีภาพ

**แนวคิด:** เฉลยนี้เลือกโทนอุ่น โดยเพิ่ม Red channel 1.3 เท่า คง Green และ Blue เดิม

**ตัวอย่าง Prompt**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน butterfly.jpg จาก dataset/
เพิ่มเฉพาะ Red channel 1.3 เท่า คง Green และ Blue เดิม
จำกัดค่าพิกเซลไม่ให้เกิน 255 แสดง Before/After โดยขนาดและรูปทรงเดิมไม่เปลี่ยน
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

![ตัวอย่างผลโจทย์เปิดไอเดีย 1](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/challenge_1.png)

**ตรวจผลงาน:** โทนภาพเปลี่ยนตามที่ระบุ ขนาดและรูปทรงเดิมไม่เปลี่ยน

[กลับสู่แบบฝึกหัดที่ 1](#exercise-1) · [สารบัญ](#contents)

<a id="example-result-2"></a>
## ผลลัพธ์ตัวอย่างที่ 2: เปรียบเทียบค่า Threshold

จำนวนแท่งขาวลดจาก 3 เป็น 2 และ 1

![ตัวอย่างผลลัพธ์ข้อ 2](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/answer_2_0.png)

![ตัวอย่างผลลัพธ์ข้อ 2](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/answer_2_1.png)

[กลับสู่ตัวอย่างที่ 2](#example-2)

<a id="solution-2"></a>
## เฉลยแบบฝึกหัดที่ 2: เลือกเฉพาะแท่งกลาง

**แนวคิด:** ตัวอย่างเลือกช่วงความสว่าง 150 ถึง 220 ค่าอื่นอาจใช้ได้หากแยกแท่งกลางได้ครบ

**ตัวอย่าง Prompt**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน graylevel.jpg จาก dataset/
เลือกพิกเซลในช่วงความสว่าง 150 ถึง 220 ให้เป็นขาว ที่เหลือดำ
แสดงต้นฉบับและ Mask เพื่อยืนยันว่าเหลือเฉพาะแท่งกลาง
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

![ตัวอย่างผลโจทย์เปิดไอเดีย 2](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/challenge_2.png)

**ตรวจผลงาน:** Mask เหลือแท่งกลางเพียงแท่งเดียว

[กลับสู่แบบฝึกหัดที่ 2](#exercise-2) · [สารบัญ](#contents)

<a id="example-result-3"></a>
## ผลลัพธ์ตัวอย่างที่ 3: เปรียบเทียบ Threshold 5 ประเภท

Binary มีขาวกับดำ ส่วนวิธีอื่นยังอาจมีระดับเทา

![ตัวอย่างผลลัพธ์ข้อ 3](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/answer_3_0.png)

[กลับสู่ตัวอย่างที่ 3](#example-3)

<a id="solution-3"></a>
## เฉลยแบบฝึกหัดที่ 3: รักษาระดับเทาของส่วนที่เลือก

**แนวคิด:** To zero inverse ที่ Threshold เท่ากับ 100 เก็บค่าด้านมืดไว้ ส่วน Binary inverse ให้พื้นที่ที่เลือกเป็นสีขาว

**ตัวอย่าง Prompt**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน graylevel.jpg จาก dataset/
ใช้ To zero inverse ที่ Threshold เท่ากับ 100
เก็บค่าพิกเซลที่ไม่เกิน 100 และเปลี่ยนค่าที่มากกว่านั้นเป็น 0 แสดงก่อนกับหลัง
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

![ตัวอย่างผลโจทย์เปิดไอเดีย 3](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/challenge_3.png)

**ตรวจผลงาน:** แท่งบนยังเป็นระดับเทาเดิม อีก 2 แท่งหายไป

[กลับสู่แบบฝึกหัดที่ 3](#exercise-3) · [สารบัญ](#contents)

<a id="example-result-4"></a>
## ผลลัพธ์ตัวอย่างที่ 4: เลือกสีและเปลี่ยนสีเฉพาะพื้นที่

แดงกับน้ำเงินรวม 6 ชิ้นถูกเลือก พื้นหลังไม่ติด Mask

![ตัวอย่างผลลัพธ์ข้อ 4](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/answer_4_0.png)

[กลับสู่ตัวอย่างที่ 4](#example-4)

<a id="solution-4"></a>
## เฉลยแบบฝึกหัดที่ 4: กำหนดสีเป้าหมายและสีเน้น

**แนวคิด:** เฉลยนี้เลือกเหลืองกับน้ำเงินและเปลี่ยนเป็นม่วง จึงมีพื้นที่ที่เปลี่ยน 5 ชิ้น

**ตัวอย่าง Prompt**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน colorobject.png จาก dataset/
วางพื้นโปร่งใสบนขาว ใช้ HSV เลือกเหลือง H=20–35 และน้ำเงิน H=100–130
ทั้งสองช่วงใช้ S=80–255, V=50–255 รวม Mask แบบ OR แล้วเปลี่ยนพื้นที่เลือกเป็นม่วง BGR=(255,0,255)
แสดงต้นฉบับ Mask และผลลัพธ์ โดยคงพิกเซลอื่นเดิม
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

![ตัวอย่างผลโจทย์เปิดไอเดีย 4](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/challenge_4.png)

**ตรวจผลงาน:** เลือกตรงกับ 2 สีที่ระบุ และเปลี่ยนสีเฉพาะพื้นที่นั้น

[กลับสู่แบบฝึกหัดที่ 4](#exercise-4) · [สารบัญ](#contents)

<a id="example-result-5"></a>
## ผลลัพธ์ตัวอย่างที่ 5: เลื่อน ย่อ หมุน และ Shear

เห็นความต่างของการเลื่อน ย่อ หมุนและเอียง

![ตัวอย่างผลลัพธ์ข้อ 5](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/answer_5_0.png)

[กลับสู่ตัวอย่างที่ 5](#example-5)

<a id="solution-5"></a>
## เฉลยแบบฝึกหัดที่ 5: จัดวางวัตถุในกรอบภาพ

**แนวคิด:** เฉลยนี้ย่อ 0.5 เท่ารอบกลางภาพ แล้วเลื่อนไปซ้าย 180 พิกเซล

**ตัวอย่าง Prompt**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน rectangle.png จาก dataset/
ย่อเนื้อภาพครึ่งหนึ่งรอบกลางภาพก่อน จากนั้นเลื่อนไปซ้าย 180 พิกเซล
คงกรอบขาว กว้าง 760 สูง 500 พิกเซล และเก็บวัตถุให้ครบ แสดงก่อนกับหลัง
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

![ตัวอย่างผลโจทย์เปิดไอเดีย 5](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/challenge_5.png)

**ตรวจผลงาน:** วัตถุเล็กลง อยู่ในตำแหน่งที่เลือก และกรอบยังเป็น 760 คูณ 500 พิกเซล

[กลับสู่แบบฝึกหัดที่ 5](#exercise-5) · [สารบัญ](#contents)

<a id="example-result-6"></a>
## ผลลัพธ์ตัวอย่างที่ 6: เปลี่ยนสี่เหลี่ยมด้านขนานเป็นจัตุรัส

ได้จัตุรัสกลางภาพ ไม่ใช่รูปด้านขนานที่หมุนแล้ว

![ตัวอย่างผลลัพธ์ข้อ 6](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/answer_6_0.png)

[กลับสู่ตัวอย่างที่ 6](#example-6)

<a id="solution-6"></a>
## เฉลยแบบฝึกหัดที่ 6: ออกแบบป้ายสี่เหลี่ยม

**แนวคิด:** เฉลยนี้เลือกขนาดประมาณ 360 คูณ 180 พิกเซล ขนาดอื่นยอมรับได้หากตรงสัดส่วนและตำแหน่ง

**ตัวอย่าง Prompt**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน rectangle2.png จาก dataset/
ใช้ Affine จับคู่ (243,110), (682,110), (80,382) ไปที่ (200,160), (560,160), (200,340) ตามลำดับ
คงกรอบขาว กว้าง 760 สูง 500 พิกเซล ให้ได้รูปน้ำเงินกว้างประมาณ 360 สูง 180 อยู่กลางภาพ
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

![ตัวอย่างผลโจทย์เปิดไอเดีย 6](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/challenge_6.png)

**ตรวจผลงาน:** ป้ายมีมุมฉาก กว้างประมาณ 2 เท่าของสูง และอยู่กลางภาพ

[กลับสู่แบบฝึกหัดที่ 6](#exercise-6) · [สารบัญ](#contents)

<a id="example-result-7"></a>
## ผลลัพธ์ตัวอย่างที่ 7: แก้มุมมองปกหนังสือ

ปกเต็มกรอบและข้อความอ่านในทิศทางเดิม

![ตัวอย่างผลลัพธ์ข้อ 7](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/answer_7_0.png)

[กลับสู่ตัวอย่างที่ 7](#example-7)

<a id="solution-7"></a>
## เฉลยแบบฝึกหัดที่ 7: จัดทำภาพปกพร้อมขอบ

**แนวคิด:** เฉลยนี้เพิ่มขอบขาว 20 พิกเซลทุกด้าน จึงได้ภาพกว้าง 340 สูง 460 พิกเซล

**ตัวอย่าง Prompt**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน left.jpg จาก dataset/
ใช้ Perspective Transformation กับจุด (280,110), (394,111), (402,292), (298,325) เรียงซ้ายบน ขวาบน ขวาล่าง ซ้ายล่าง
ทำภาพปกกว้าง 300 สูง 420 พิกเซล แล้วเพิ่มขอบขาว 20 พิกเซลทุกด้าน
รายงานขนาดภาพสุดท้าย โดยไม่กลับด้านข้อความหรือยืดปก
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

![ตัวอย่างผลโจทย์เปิดไอเดีย 7](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/challenge_7.png)

**ตรวจผลงาน:** ปกไม่บิดหรือกลับด้าน ขอบตรงตามที่ระบุ และขนาดภาพเพิ่มตามความกว้างขอบ

[กลับสู่แบบฝึกหัดที่ 7](#exercise-7) · [สารบัญ](#contents)

<a id="example-result-8"></a>
## ผลลัพธ์ตัวอย่างที่ 8: Crop ลูกบอลและ Resize โดยคงสัดส่วน

บอลอยู่ครบและไม่ยืดเป็นวงรี

![ตัวอย่างผลลัพธ์ข้อ 8](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/answer_8_0.png)

[กลับสู่ตัวอย่างที่ 8](#example-8)

<a id="solution-8"></a>
## เฉลยแบบฝึกหัดที่ 8: เตรียมภาพบอลแนวนอน

**แนวคิด:** เฉลยนี้ทำภาพบอลเป็น 300 คูณ 300 ก่อน แล้วเติมขอบขาวซ้ายและขวาด้านละ 50 พิกเซล

**ตัวอย่าง Prompt**

```text
สร้างโค้ด Python สำหรับ Google Colab/Jupyter Notebook โดยใช้ OpenCV และ Matplotlib
อ่าน messi.jpg จาก dataset/
ครอป x=330 ถึงก่อน 395 และ y=280 ถึงก่อน 342
เติมขอบให้เป็นจัตุรัสแล้ว Resize เป็น กว้าง 300 สูง 300 พิกเซล จากนั้นเติมขอบขาวซ้ายและขวาข้างละ 50
ให้ผลกว้าง 400 สูง 300 โดยบอลไม่ยืด
แสดงภาพในโน้ตบุ๊กและอธิบายผลเป็นภาษาไทยไม่เกิน 2 ประโยค
```

![ตัวอย่างผลโจทย์เปิดไอเดีย 8](https://raw.githubusercontent.com/bluebox-dev/Computer-Vision-Al/main/assets/notebook/challenge_8.png)

**ตรวจผลงาน:** ผลลัพธ์กว้าง 400 สูง 300 พิกเซล ลูกบอลอยู่ครบและไม่ยืด

[กลับสู่แบบฝึกหัดที่ 8](#exercise-8) · [สารบัญ](#contents)

<a id="credits"></a>
# เครดิตและแนวทางเรียนรู้เพิ่มเติม

**เครดิตสไลด์ต้นฉบับ**

Asst. Prof. Dr. Anakkapon Saenthon, King Mongkut’s Institute of Technology Ladkrabang (KMITL). *COMPUTER VISIONS, Course Code 01416500: Basic Image Processing / Feature Extraction และ Image Transformation*. ไฟล์ **Day1_1_BasicImageProcessing.pdf**, 72 หน้า

เอกสารนี้เรียบเรียงหลักการและตัวอย่างจากสไลด์ดังกล่าว ภาพสไลด์ประกอบคงเครดิตต้นฉบับไว้ ส่วน Prompt แบบฝึกหัดประยุกต์ และภาพเปรียบเทียบจัดทำเพิ่มจาก dataset ของผู้เข้าอบรม

**การปรับสำหรับชุดข้อมูล:** ใช้ left.jpg แทน right.jpg ในโจทย์หนังสือ พิกัดและค่าทดลองปรับให้ตรงภาพจริง สูตรเฉลี่ยและเงื่อนไข Threshold ใช้ตามการทำงานของ OpenCV

**เรียนรู้เพิ่มเติมจากเอกสาร OpenCV**

| หัวข้อ | แหล่งเรียนรู้ | สิ่งที่ควรทดลองต่อ |
|---|---|---|
| ระบบสีและ HSV | [Changing Colorspaces](https://docs.opencv.org/4.13.0/df/d9d/tutorial_py_colorspaces.html) | ใช้ภาพวัตถุของตนเอง แล้วปรับช่วงสี |
| Threshold | [Image Thresholding](https://docs.opencv.org/4.13.0/d7/d4d/tutorial_py_thresholding.html) | เปรียบเทียบ Global กับ Adaptive Threshold เมื่อแสงไม่สม่ำเสมอ |
| การแปลงภาพ | [Geometric Transformations](https://docs.opencv.org/4.13.0/da/d6e/tutorial_py_geometric_transformations.html) | เปลี่ยนจุดอ้างอิงและขนาดผลลัพธ์ แล้วสังเกตความแตกต่าง |

เลือกทดลองครั้งละ 1 ปัจจัย เก็บ Prompt และภาพ Before/After พร้อมสรุปว่าอะไรทำให้ผลดีขึ้น

[กลับสู่สารบัญ](#contents)